## Day 1 

In [1]:
import os
import glob

base_input_path = "/kaggle/input"
subfolders = [f.path for f in os.scandir(base_input_path) if f.is_dir()]

if subfolders:
    INPUT_DIR = subfolders[0]
    print(f"Main Dataset Folder Located at: {INPUT_DIR}")
    print(f"Verified directory presence: {os.path.exists(INPUT_DIR)}")
else:
    print("CRITICAL ERROR: No datasets attached. Please click '+ Add Input' in the top right and add 'rohanrao/nifty50-stock-market-data'")

OUTPUT_DIR = "/kaggle/working/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Main Dataset Folder Located at: /kaggle/input/datasets
Verified directory presence: True


In [2]:
all_csv_paths = glob.glob(os.path.join(INPUT_DIR, "**", "*.csv"), recursive=True)

EXCLUDED_KEYWORDS = ["all", "metadata", "index", "stock_metadata"]
stock_files = [
    f for f in all_csv_paths 
    if not any(keyword in os.path.basename(f).lower() for keyword in EXCLUDED_KEYWORDS)
]

print(f"Total individual stock files discovered: {len(stock_files)}")
print("-" * 50)

if len(stock_files) > 0:
    print("Sample files successfully indexed and verified:")
    for f in sorted(stock_files)[:8]:
        print(f" - {os.path.basename(f)} (Path: {os.path.relpath(f, INPUT_DIR)})")
else:
    print("WARNING: Files still not found. Printing all files in the input directory for diagnostics:")
    for root, dirs, files in os.walk(INPUT_DIR):
        for file in files[:10]:
            print(os.path.join(root, file))

Total individual stock files discovered: 50
--------------------------------------------------
Sample files successfully indexed and verified:
 - ADANIPORTS.csv (Path: rohanrao/nifty50-stock-market-data/ADANIPORTS.csv)
 - ASIANPAINT.csv (Path: rohanrao/nifty50-stock-market-data/ASIANPAINT.csv)
 - AXISBANK.csv (Path: rohanrao/nifty50-stock-market-data/AXISBANK.csv)
 - BAJAJ-AUTO.csv (Path: rohanrao/nifty50-stock-market-data/BAJAJ-AUTO.csv)
 - BAJAJFINSV.csv (Path: rohanrao/nifty50-stock-market-data/BAJAJFINSV.csv)
 - BAJFINANCE.csv (Path: rohanrao/nifty50-stock-market-data/BAJFINANCE.csv)
 - BHARTIARTL.csv (Path: rohanrao/nifty50-stock-market-data/BHARTIARTL.csv)
 - BPCL.csv (Path: rohanrao/nifty50-stock-market-data/BPCL.csv)


In [3]:
processed_frames = []
REQUIRED_COLUMNS = ['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']

print("Starting extraction and structural alignment engine...\n")

for index, file_path in enumerate(stock_files):
    ticker = os.path.splitext(os.path.basename(file_path))[0]
    try:
        # Load the stock data frame
        df = pd.read_csv(file_path)
        
        # 1. Isolate standard equity shares, filtering out derivative options/series
        if 'Series' in df.columns:
            df = df[df['Series'] == 'EQ']
            
        if df.empty:
            continue
            
        # 2. Standardize data types & pull only vital architecture features
        df['Date'] = pd.to_datetime(df['Date'])
        df = df[REQUIRED_COLUMNS].copy()
        
        # 3. Sort chronologically to prevent lookahead target leakage
        df = df.sort_values('Date').reset_index(drop=True)
        
        # 4. Fill gaps (Forward fill price from previous active close, backward fill gaps at start)
        df[['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].ffill().bfill()
        df['Volume'] = df['Volume'].fillna(0)
        
        # 5. Synchronize the timeline window (2010 to 2020) for dense portfolio matrix calculations
        df = df[(df['Date'] >= '2010-01-01') & (df['Date'] <= '2020-12-31')]
        
        # 6. Ensure deep historical coverage (drop assets lacking standard continuity)
        if len(df) > 1500:
            processed_frames.append(df)
            if (len(processed_frames) % 10 == 0) or (index == len(stock_files) - 1):
                print(f"Status Update: Successfully aligned and cached {len(processed_frames)} assets...")
                
    except Exception as e:
        print(f"Skipping processing matrix for {ticker} due to error: {e}")

# Merge everything into a structured master base file
if processed_frames:
    master_df = pd.concat(processed_frames, ignore_index=True)
    master_df = master_df.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)
    print("\n--- Preprocessing Engine Complete ---")
    print(f"Aggregated a total of {master_df.shape[0]} daily row data entries across {master_df['Symbol'].nunique()} clean tickers.")
else:
    print("CRITICAL ERROR: Processing resulted in an empty frame. Check date-range boundary configurations.")

Starting extraction and structural alignment engine...

Status Update: Successfully aligned and cached 10 assets...
Status Update: Successfully aligned and cached 20 assets...
Status Update: Successfully aligned and cached 30 assets...
Status Update: Successfully aligned and cached 40 assets...
Status Update: Successfully aligned and cached 49 assets...

--- Preprocessing Engine Complete ---
Aggregated a total of 133554 daily row data entries across 56 clean tickers.


In [4]:
import os

# 1. Confirm absence of null artifacts
null_matrix = master_df.isnull().sum()
print("Data Integrity Report (Missing Values Count per Column):")
print(null_matrix)
print("-" * 50)

# 2. Display an overview of the structural layout to verify columns match exactly
print("Master Base Table - First 5 Clean Records:")
print(master_df.head())
print("-" * 50)

# 3. Save to output directory
final_output_path = os.path.join(OUTPUT_DIR, "processed_base.csv")
master_df.to_csv(final_output_path, index=False)

print(f"SUCCESS: Day 1 core framework completed!")
print(f"Saved structural master artifact to: {final_output_path}")

Data Integrity Report (Missing Values Count per Column):
Date      0
Symbol    0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
--------------------------------------------------
Master Base Table - First 5 Clean Records:
        Date      Symbol   Open    High     Low   Close   Volume
0 2012-01-17  ADANIPORTS  137.1  141.00  135.00  140.00  1636196
1 2012-01-18  ADANIPORTS  142.0  143.80  138.70  141.70   890591
2 2012-01-19  ADANIPORTS  144.0  150.55  143.15  149.40  1456077
3 2012-01-20  ADANIPORTS  151.9  157.60  150.25  155.40  1634070
4 2012-01-23  ADANIPORTS  155.4  155.40  145.10  146.75  1657609
--------------------------------------------------
SUCCESS: Day 1 core framework completed!
Saved structural master artifact to: /kaggle/working/data/processed_base.csv


## Day 2

In [5]:
import os
import numpy as np
import pandas as pd

# Define data paths pointing to your Day 1 outputs
DATA_DIR = "/kaggle/working/data"
input_path = os.path.join(DATA_DIR, "processed_base.csv")

print("Loading preprocessed base matrix...")
base_df = pd.read_csv(input_path)
base_df['Date'] = pd.to_datetime(base_df['Date'])

# Helper Function: Calculate RSI natively without requiring external heavy packages
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9) # Prevent division by zero
    return 100 - (100 / (1 + rs))

print(f"Base data loaded successfully. Total rows to process: {len(base_df)}")

Loading preprocessed base matrix...
Base data loaded successfully. Total rows to process: 133554


In [6]:
processed_ticker_groups = []

# Group data by individual unique symbols to avoid inter-asset indicator contamination
ticker_groups = base_df.groupby('Symbol')

print("Beginning engineering run across all index assets...\n")

for symbol, group in ticker_groups:
    # Always sort chronologically within the group
    df = group.sort_values('Date').copy()
    
    # 1. Base Metrics
    df['daily_return'] = df['Close'].pct_change()
    
    # 2. Moving Averages (Trend Tracking)
    df['SMA10'] = df['Close'].rolling(window=10).mean()
    df['SMA20'] = df['Close'].rolling(window=20).mean()
    df['SMA50'] = df['Close'].rolling(window=50).mean()
    df['EMA20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # 3. Momentum Vector Math (RSI & MACD)
    df['RSI14'] = calculate_rsi(df['Close'], period=14)
    
    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # 4. Volatility Channels (Standard Deviation & Bollinger Bands)
    df['volatility_20d'] = df['daily_return'].rolling(window=20).std()
    rolling_std_close = df['Close'].rolling(window=20).std()
    df['Upper_Band'] = df['SMA20'] + (rolling_std_close * 2)
    df['Lower_Band'] = df['SMA20'] - (rolling_std_close * 2)
    
    # 5. Competition Target Vector Definition (5-Day Future Direction Allocation)
    # 1 if price 5 days from now is higher than today's close, else 0
    df['future_direction'] = (df['Close'].shift(-5) > df['Close']).astype(int)
    
    # 6. Technical Window Truncation
    # Drop rows that don't have enough history to compute indicators (like SMA50)
    # The last 5 rows per asset are safely dropped since their future target doesn't exist yet
    df = df.dropna().reset_index(drop=True)
    
    processed_ticker_groups.append(df)

# Recombine all asset feature spaces into a master training set matrix
feature_df = pd.concat(processed_ticker_groups, ignore_index=True)
feature_df = feature_df.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)

print("\n--- Feature Pipeline Checkpoint Reached ---")
print(f"Total structured row profiles: {feature_df.shape[0]} across {feature_df['Symbol'].nunique()} active symbols.")

Beginning engineering run across all index assets...


--- Feature Pipeline Checkpoint Reached ---
Total structured row profiles: 130810 across 56 active symbols.


In [7]:
# 1. Check for any infinite values or stray null elements inside the operational column blocks
target_features = ['daily_return', 'SMA10', 'SMA20', 'SMA50', 'EMA20', 'RSI14', 'MACD', 'Signal', 'volatility_20d', 'Upper_Band', 'Lower_Band']
null_summary = feature_df[target_features].isnull().sum()

print("Feature Space Validation Mapping (Null Count):")
print(null_summary)
print("-" * 60)

# 2. Check the distribution balance of your target labels (Up vs Down)
distribution = feature_df['future_direction'].value_counts(normalize=True) * 100
print("Target Direction Balance Metric:")
print(f" -> Class 0 (Down/Flat Over Next 5 Days): {distribution[0]:.2f}%")
print(f" -> Class 1 (Upward Shift Over Next 5 Days): {distribution[1]:.2f}%")
print("-" * 60)

# 3. Write final dataset table out to disk
output_feature_path = os.path.join(DATA_DIR, "feature_dataset.csv")
feature_df.to_csv(output_feature_path, index=False)

print(f"SUCCESS: Day 2 Feature Dataset constructed and verified!")
print(f"Saved working configuration matrix to: {output_feature_path}")

Feature Space Validation Mapping (Null Count):
daily_return      0
SMA10             0
SMA20             0
SMA50             0
EMA20             0
RSI14             0
MACD              0
Signal            0
volatility_20d    0
Upper_Band        0
Lower_Band        0
dtype: int64
------------------------------------------------------------
Target Direction Balance Metric:
 -> Class 0 (Down/Flat Over Next 5 Days): 47.44%
 -> Class 1 (Upward Shift Over Next 5 Days): 52.56%
------------------------------------------------------------
SUCCESS: Day 2 Feature Dataset constructed and verified!
Saved working configuration matrix to: /kaggle/working/data/feature_dataset.csv


## Day 3

In [8]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Define directory structures
DATA_DIR = "/kaggle/working/data"
MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

input_feature_path = os.path.join(DATA_DIR, "feature_dataset.csv")
print("Reading feature matrix engine from disk...")
df = pd.read_csv(input_feature_path)
df['Date'] = pd.to_datetime(df['Date'])

# 1. Define exact operational feature vectors and model target
FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'SMA10', 'SMA20', 'SMA50', 'EMA20', 
    'RSI14', 'MACD', 'Signal', 'volatility_20d', 'Upper_Band', 'Lower_Band'
]
TARGET_COLUMN = 'future_direction'

# 2. Chronological Backtest Splitting (Prevents look-ahead financial data leakage)
train_mask = df['Date'] <= '2018-12-31'
test_mask = df['Date'] >= '2019-01-01'

X_train = df.loc[train_mask, FEATURE_COLUMNS]
y_train = df.loc[train_mask, TARGET_COLUMN]

X_test = df.loc[test_mask, FEATURE_COLUMNS]
y_test = df.loc[test_mask, TARGET_COLUMN]

print(f"Matrix configurations structured successfully:")
print(f" -> Training Allocation Size (2010-2018): {X_train.shape[0]} samples")
print(f" -> Evaluation Allocation Size (2019-2020): {X_test.shape[0]} samples")

Reading feature matrix engine from disk...
Matrix configurations structured successfully:
 -> Training Allocation Size (2010-2018): 106457 samples
 -> Evaluation Allocation Size (2019-2020): 24353 samples


In [9]:
print("Initializing Random Forest Classifier Engine...")

# Using structured hyperparameter caps optimized for multi-asset market data tables
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,  # Force parallel calculation across all Kaggle CPU cores
    verbose=1
)

print("Training predictive engine on training matrix...")
model.fit(X_train, y_train)
print("Training execution finalized.")

# Save frozen model artifact for Streamlit inference pipeline integration
model_output_path = os.path.join(MODELS_DIR, "stock_predictor.pkl")
joblib.dump(model, model_output_path)
print(f"SUCCESS: Model pipeline serialized and saved to: {model_output_path}")

Initializing Random Forest Classifier Engine...
Training predictive engine on training matrix...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   12.9s


Training execution finalized.
SUCCESS: Model pipeline serialized and saved to: /kaggle/working/models/stock_predictor.pkl


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.5s finished


In [10]:
print("Generating out-of-sample forward predictions...")
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 1. Compute Directional Performance Metrics
acc_score = accuracy_score(y_test, y_pred)
print("\n" + "="*20 + " PREDICTOR EVALUATION METRICS " + "="*20)
print(f"Out-of-Sample Directional Accuracy: {acc_score * 100:.2f}%")
print("\nClassification Matrix Metrics Breakdown:")
print(classification_report(y_test, y_pred, target_names=['DOWN/FLAT', 'UP']))

# 2. Extract Feature Importances (Fulfills the Platform's Explainability Mandate)
importances = model.feature_importances_
feature_ranking = pd.DataFrame({
    'Feature_Indicator': FEATURE_COLUMNS,
    'Importance_Score': importances
}).sort_values(by='Importance_Score', ascending=False).reset_index(drop=True)

print("\n" + "="*21 + " FEATURE EXPLAINABILITY LOGS " + "="*21)
for rank, row in feature_ranking.iterrows():
    print(f" Rank {rank+1:02d} | {row['Feature_Indicator']:<15} | Weight: {row['Importance_Score']*100:.2f}%")
print("="*72)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s


Generating out-of-sample forward predictions...

==================== PREDICTOR EVALUATION METRICS ====================
Out-of-Sample Directional Accuracy: 49.66%

Classification Matrix Metrics Breakdown:
              precision    recall  f1-score   support

   DOWN/FLAT       0.47      0.32      0.38     11807
          UP       0.51      0.66      0.57     12546

    accuracy                           0.50     24353
   macro avg       0.49      0.49      0.48     24353
weighted avg       0.49      0.50      0.48     24353



[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished



===================== FEATURE EXPLAINABILITY LOGS =====================
 Rank 01 | volatility_20d  | Weight: 8.28%
 Rank 02 | Volume          | Weight: 7.64%
 Rank 03 | RSI14           | Weight: 7.62%
 Rank 04 | Signal          | Weight: 7.45%
 Rank 05 | SMA50           | Weight: 7.06%
 Rank 06 | MACD            | Weight: 6.69%
 Rank 07 | Lower_Band      | Weight: 6.66%
 Rank 08 | Upper_Band      | Weight: 6.34%
 Rank 09 | daily_return    | Weight: 6.12%
 Rank 10 | SMA20           | Weight: 5.80%
 Rank 11 | EMA20           | Weight: 5.77%
 Rank 12 | SMA10           | Weight: 5.47%
 Rank 13 | Close           | Weight: 5.20%
 Rank 14 | High            | Weight: 4.94%
 Rank 15 | Low             | Weight: 4.63%
 Rank 16 | Open            | Weight: 4.36%


## Day 4

In [11]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/working/data"
input_feature_path = os.path.join(DATA_DIR, "feature_dataset.csv")

print("Loading data for Risk Assessment engine...")
df = pd.read_csv(input_feature_path)
df['Date'] = pd.to_datetime(df['Date'])

# Configuration parameters
RISK_FREE_RATE = 0.06  # 6% Risk-Free Rate baseline standard for Indian Markets
TRADING_DAYS_PER_YEAR = 252

risk_records = []
ticker_groups = df.groupby('Symbol')

print("Computing quantitative risk profiles across all active listings...")

for symbol, group in ticker_groups:
    group = group.sort_values('Date')
    returns = group['daily_return'].values
    prices = group['Close'].values
    
    if len(returns) < 100:
        continue
        
    # 1. Volatility Calculation (Annualized)
    daily_vol = np.std(returns)
    annualized_vol = daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)
    
    # 2. CAGR Calculation
    total_days = (group['Date'].max() - group['Date'].min()).days
    years = total_days / 365.25
    initial_val = prices[0]
    final_val = prices[-1]
    cagr = (final_val / initial_val) ** (1 / years) - 1 if initial_val > 0 else 0
    
    # 3. Sharpe Ratio
    excess_return = cagr - RISK_FREE_RATE
    sharpe_ratio = excess_return / annualized_vol if annualized_vol > 0 else 0
    
    # 4. Sortino Ratio (Isolating downside semi-deviation)
    downside_returns = returns[returns < 0]
    downside_vol = np.std(downside_returns) * np.sqrt(TRADING_DAYS_PER_YEAR) if len(downside_returns) > 0 else 1e-9
    sortino_ratio = excess_return / downside_vol if downside_vol > 0 else 0
    
    # 5. Maximum Drawdown Calculation
    # Compute rolling cumulative returns to track historical peaks
    cum_returns = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cum_returns)
    drawdowns = (cum_returns - running_max) / running_max
    max_drawdown = np.min(drawdowns) if len(drawdowns) > 0 else 0
    
    risk_records.append({
        'Symbol': symbol,
        'Annualized_Volatility': annualized_vol,
        'CAGR': cagr,
        'Sharpe_Ratio': sharpe_ratio,
        'Sortino_Ratio': sortino_ratio,
        'Max_Drawdown': max_drawdown
    })

risk_base_df = pd.DataFrame(risk_records)
print(f"Risk metrics extracted successfully for {len(risk_base_df)} distinct corporate listings.")

Loading data for Risk Assessment engine...
Computing quantitative risk profiles across all active listings...
Risk metrics extracted successfully for 56 distinct corporate listings.


In [12]:
print("Normalizing metrics into standard Risk Scores (0-100)...")

# Define operational metrics for composite mapping ranking
# High Volatility and Deep Drawdowns increase Risk Score
# High Sharpe reduces Risk Score
vol_rank = risk_base_df['Annualized_Volatility'].rank(pct=True)
dd_rank = risk_base_df['Max_Drawdown'].abs().rank(pct=True)
sharpe_rank = risk_base_df['Sharpe_Ratio'].rank(pct=True, ascending=False)

# Composite mathematical ranking matrix weights
composite_rank = (vol_rank * 0.4) + (dd_rank * 0.4) + (sharpe_rank * 0.2)

# Map precisely between 0 and 100 boundaries
risk_base_df['Risk_Score'] = (composite_rank * 100).round(1)

# Apply deterministic institutional grading bounds
def assign_risk_category(score):
    if score <= 30.0:
        return 'Low Risk'
    elif score <= 60.0:
        return 'Medium Risk'
    else:
        return 'High Risk'

risk_base_df['Risk_Category'] = risk_base_df['Risk_Score'].apply(assign_risk_category)

print("\nRisk Stratification Breakdown Summary:")
print(risk_base_df['Risk_Category'].value_counts())
print("-" * 60)

# Save the final structural asset analytics output to disk
risk_output_path = os.path.join(DATA_DIR, "risk_metrics.csv")
risk_base_df.to_csv(risk_output_path, index=False)

print(f"SUCCESS: Day 4 Risk Analytics layer locked down.")
print(f"Metrics portfolio artifact safely stored at: {risk_output_path}")

Normalizing metrics into standard Risk Scores (0-100)...

Risk Stratification Breakdown Summary:
Risk_Category
High Risk      23
Medium Risk    18
Low Risk       15
Name: count, dtype: int64
------------------------------------------------------------
SUCCESS: Day 4 Risk Analytics layer locked down.
Metrics portfolio artifact safely stored at: /kaggle/working/data/risk_metrics.csv


## Day 5

In [13]:
# Install PyPortfolioOpt seamlessly in the background
!pip install PyPortfolioOpt --quiet
print("SUCCESS: PyPortfolioOpt library successfully installed and integrated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 6.7 MB/s eta 0:00:00
SUCCESS: PyPortfolioOpt library successfully installed and integrated.


In [14]:
import os
import numpy as np
import pandas as pd
from pypfopt import expected_returns, risk_models, EfficientFrontier

DATA_DIR = "/kaggle/working/data"
base_data_path = os.path.join(DATA_DIR, "processed_base.csv")

print("Loading baseline transaction matrix...")
base_df = pd.read_csv(base_data_path)
base_df['Date'] = pd.to_datetime(base_df['Date'])

# Pivot the data frame layout to create a clean daily closing price matrix per asset
print("Pivoting data structure into price-series matrix...")
price_matrix = base_df.pivot(index='Date', columns='Symbol', values='Close')

# Handle any remaining asset operational boundary gaps via forward fill
price_matrix = price_matrix.ffill().bfill()

print(f"Price matrix established: {price_matrix.shape[0]} tracking dates across {price_matrix.shape[1]} unique ticker tickers.")

# Calculate mathematical inputs for Modern Portfolio Theory
print("Calculating historical expected returns and covariance matrix...")
mu = expected_returns.mean_historical_return(price_matrix, frequency=252)
S = risk_models.sample_cov(price_matrix, frequency=252)
print("Statistical parameters compiled.")

Loading baseline transaction matrix...
Pivoting data structure into price-series matrix...
Price matrix established: 2730 tracking dates across 56 unique ticker tickers.
Calculating historical expected returns and covariance matrix...
Statistical parameters compiled.


In [15]:
def save_cleaned_weights(raw_weights, output_filename):
    """Filters out tiny asset weights (<1%) and normalizes remaining allocation targets."""
    cleaned = {ticker: weight for ticker, weight in raw_weights.items() if weight >= 0.01}
    total_weight = sum(cleaned.values())
    
    # Normalize weights so they sum to exactly 100%
    normalized = {ticker: round(weight / total_weight, 4) for ticker, weight in cleaned.items()}
    
    df_weights = pd.DataFrame(list(normalized.items()), columns=['Symbol', 'Allocation_Weight'])
    df_weights = df_weights.sort_values(by='Allocation_Weight', ascending=False).reset_index(drop=True)
    
    out_path = os.path.join(DATA_DIR, output_filename)
    df_weights.to_csv(out_path, index=False)
    return df_weights

print("--- Starting Portfolio Optimization Run --- \n")

# 1. CONSERVATIVE PROFILE: Minimum Variance Frontier Target
print("Optimizing Conservative Allocation Profile (Minimum Volatility Target)...")
ef_cons = EfficientFrontier(mu, S)
raw_weights_cons = ef_cons.min_volatility()
df_cons = save_cleaned_weights(raw_weights_cons, "conservative_portfolio.csv")
print(f" -> Conservative Profile complete. Assets selected: {len(df_cons)}")

# 2. BALANCED PROFILE: Maximum Sharpe Sharpe Optimization
print("Optimizing Balanced Allocation Profile (Maximum Sharpe Ratio Target)...")
ef_bal = EfficientFrontier(mu, S)
raw_weights_bal = ef_bal.max_sharpe(risk_free_rate=0.06)
df_bal = save_cleaned_weights(raw_weights_bal, "balanced_portfolio.csv")
print(f" -> Balanced Profile complete. Assets selected: {len(df_bal)}")

# 3. AGGRESIVE PROFILE: Maximizing Quadratic Utility 
print("Optimizing Aggressive Allocation Profile (Max Quadratic Utility Target)...")
ef_agg = EfficientFrontier(mu, S)
# High risk tolerance parameter (risk_averion=1.5) to prioritize higher raw returns
raw_weights_agg = ef_agg.max_quadratic_utility(risk_aversion=1.5)
df_agg = save_cleaned_weights(raw_weights_agg, "aggressive_portfolio.csv")
print(f" -> Aggressive Profile complete. Assets selected: {len(df_agg)}")

print("\n" + "="*50)
print("SUCCESS: Day 5 Asset Allocations complete!")
print("Portfolio asset files saved to disk. Ready for Recommendation synthesis.")
print("="*50)

--- Starting Portfolio Optimization Run --- 

Optimizing Conservative Allocation Profile (Minimum Volatility Target)...
 -> Conservative Profile complete. Assets selected: 8
Optimizing Balanced Allocation Profile (Maximum Sharpe Ratio Target)...
 -> Balanced Profile complete. Assets selected: 6
Optimizing Aggressive Allocation Profile (Max Quadratic Utility Target)...
 -> Aggressive Profile complete. Assets selected: 3

SUCCESS: Day 5 Asset Allocations complete!
Portfolio asset files saved to disk. Ready for Recommendation synthesis.


## Day 6


In [18]:
import os
import joblib
import pandas as pd
import numpy as np

# 1. Coordinate and declare active paths
DATA_DIR = "/kaggle/working/data"
MODELS_DIR = "/kaggle/working/models"

print("Step 1: Reading core engine files from workspace cache...")
df_features = pd.read_csv(os.path.join(DATA_DIR, "feature_dataset.csv"))
df_risk = pd.read_csv(os.path.join(DATA_DIR, "risk_metrics.csv"))
model = joblib.load(os.path.join(MODELS_DIR, "stock_predictor.pkl"))

# Model training column references
FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'SMA10', 'SMA20', 'SMA50', 'EMA20', 
    'RSI14', 'MACD', 'Signal', 'volatility_20d', 'Upper_Band', 'Lower_Band'
]

# 2. Extract current market snapshot profile boundaries
print("Step 2: Isolating most recent chronological records per asset...")
latest_market_states = df_features.sort_values('Date').groupby('Symbol').last().reset_index()

# 3. Compute batch predictive metrics 
print("Step 3: Calculating 5-day predictive momentum arrays...")
X_inference = latest_market_states[FEATURE_COLUMNS]
predictions = model.predict(X_inference)
probabilities = model.predict_proba(X_inference)[:, 1]

# 4. Attach raw model metadata to our snapshot frame
latest_market_states['ml_prediction'] = predictions
latest_market_states['ml_confidence'] = probabilities

# 5. Merge ML findings with static quantitative risk calculations from Day 4
unified_recommendation_base = pd.merge(
    latest_market_states[['Symbol', 'Close', 'ml_prediction', 'ml_confidence']], 
    df_risk[['Symbol', 'Annualized_Volatility', 'Sharpe_Ratio', 'Risk_Score', 'Risk_Category']], 
    on='Symbol'
)

print(f"\nUnified intelligence matrix established for {len(unified_recommendation_base)} assets.")

Step 1: Reading core engine files from workspace cache...
Step 2: Isolating most recent chronological records per asset...
Step 3: Calculating 5-day predictive momentum arrays...

Unified intelligence matrix established for 56 assets.


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


In [19]:
recommendations = []
justifications = []

print("Step 6: Synthesizing final execution actions based on risk-first matrix rules...")

for idx, row in unified_recommendation_base.iterrows():
    pred = row['ml_prediction']
    conf = row['ml_confidence']
    score = row['Risk_Score']
    sharpe = row['Sharpe_Ratio']
    
    # Apply our conditional risk-adjusted logic tree
    if pred == 1 and score < 60.0:
        action = "BUY"
        reason = (f"Strong near-term upward momentum detected (ML Confidence: {conf*100:.1f}%). "
                  f"Asset displays a healthy historical Sharpe Ratio of {sharpe:.2f} within safe risk boundaries.")
        
    elif pred == 1 and score >= 60.0:
        action = "HOLD"
        reason = (f"Upward trend spotted by ML model (Confidence: {conf*100:.1f}%), "
                  f"but historical Volatility index is high (Risk Score: {score:.1f}). "
                  f"Deferring aggressive allocation to prevent downstream drawdown risks.")
        
    else:  # ML predicts flat or downward movement over the next 5 days (pred == 0)
        action = "SELL"
        reason = (f"Negative trend indicators or flat structural performance expected over the next 5-day horizon. "
                  f"Risk Score stands at {score:.1f}. Pruning portfolio exposure is recommended.")
        
    recommendations.append(action)
    justifications.append(reason)

# Attach compiled logic arrays back to the master dataframe
unified_recommendation_base['Action_Item'] = recommendations
unified_recommendation_base['Quantitative_Justification'] = justifications

# Rearrange columns visually for clear presentation layers in Streamlit
final_layout = [
    'Symbol', 'Close', 'ml_prediction', 'ml_confidence', 
    'Risk_Score', 'Risk_Category', 'Action_Item', 'Quantitative_Justification'
]
output_recommendation_df = unified_recommendation_base[final_layout].copy()
output_recommendation_df = output_recommendation_df.sort_values(by='Risk_Score').reset_index(drop=True)

print("\nUnified Platform Directive Breakdown Summary:")
print(output_recommendation_df['Action_Item'].value_counts())
print("-" * 65)

# Export persistent target file back into the workspace directory
rec_output_path = os.path.join(DATA_DIR, "investment_recommendations.csv")
output_recommendation_df.to_csv(rec_output_path, index=False)

print(f"SUCCESS: 'investment_recommendations.csv' has been generated!")
print(f"Verified File Location: {rec_output_path}")

Step 6: Synthesizing final execution actions based on risk-first matrix rules...

Unified Platform Directive Breakdown Summary:
Action_Item
SELL    32
BUY     15
HOLD     9
Name: count, dtype: int64
-----------------------------------------------------------------
SUCCESS: 'investment_recommendations.csv' has been generated!
Verified File Location: /kaggle/working/data/investment_recommendations.csv


## Additional Improvements to make the model better

In [20]:
# 1. Background Environment Patching
!pip install lightgbm --quiet

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report
import joblib

DATA_DIR = "/kaggle/working/data"
MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Loading engineered feature datasets...")
df = pd.read_csv(os.path.join(DATA_DIR, "feature_dataset.csv"))
df['Date'] = pd.to_datetime(df['Date'])

# Define our core analytical feature columns
FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'SMA10', 'SMA20', 'SMA50', 'EMA20', 
    'RSI14', 'MACD', 'Signal', 'volatility_20d', 'Upper_Band', 'Lower_Band'
]

# Calculate a continuous regression target variable: 5-Day Forward Percentage Return
df['future_return'] = df.groupby('Symbol')['Close'].shift(-5) / df['Close'] - 1

# Safely drop boundary artifacts created by the forward shifts
df = df.dropna(subset=['future_return', 'future_direction']).reset_index(drop=True)

# Chronological Train/Test Partitioning
train_mask = df['Date'] <= '2018-12-31'
test_mask = df['Date'] >= '2019-01-01'

X_train, y_train_cls, y_train_reg = df.loc[train_mask, FEATURE_COLUMNS], df.loc[train_mask, 'future_direction'], df.loc[train_mask, 'future_return']
X_test, y_test_cls, y_test_reg = df.loc[test_mask, FEATURE_COLUMNS], df.loc[test_mask, 'future_direction'], df.loc[test_mask, 'future_return']

print(f"Matrix partitions locked. Training size: {X_train.shape[0]} | Testing size: {X_test.shape[0]}")

Loading engineered feature datasets...
Matrix partitions locked. Training size: 106422 | Testing size: 24108


In [21]:
print("--- Training Engine Alpha: LightGBM Gradient Classifier ---")
cls_model = lgb.LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1)
cls_model.fit(X_train, y_train_cls)

print("--- Training Engine Beta: LightGBM Gradient Regressor ---")
reg_model = lgb.LGBMRegressor(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1)
reg_model.fit(X_train, y_train_reg)

# Save both specialized model payloads to disk
joblib.dump(cls_model, os.path.join(MODELS_DIR, "lgb_classifier.pkl"))
joblib.dump(reg_model, os.path.join(MODELS_DIR, "lgb_regressor.pkl"))
print("\nPredictive model payloads successfully frozen and serialized.")

# Generate validation predictions
y_pred_cls = cls_model.predict(X_test)
y_pred_reg = reg_model.predict(X_test)

# Compute performance metrics requested by the guidelines
acc = accuracy_score(y_test_cls, y_pred_cls)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n" + "="*20 + " METRIC COMPLIANCE REPORT " + "="*20)
print(f"Directional Turn Accuracy : {acc * 100:.2f}%")
print(f"Mean Absolute Error (MAE) : {mae:.5f} (Raw return deviation bounds)")
print(f"R² Performance Score      : {r2:.5f}")
print("="*66)

--- Training Engine Alpha: LightGBM Gradient Classifier ---
--- Training Engine Beta: LightGBM Gradient Regressor ---

Predictive model payloads successfully frozen and serialized.

==================== METRIC COMPLIANCE REPORT ====================
Directional Turn Accuracy : 49.88%
Mean Absolute Error (MAE) : 0.04055 (Raw return deviation bounds)
R² Performance Score      : -0.07093


In [22]:
print("Deploying Statistical Anomaly Detection layers...")

anomaly_records = []
ticker_groups = df.groupby('Symbol')

for symbol, group in ticker_groups:
    group = group.sort_values('Date').copy()
    
    # Calculate rolling statistical baselines (50-day windows)
    rolling_vol_mean = group['daily_return'].rolling(window=50).mean()
    rolling_vol_std = group['daily_return'].rolling(window=50).std()
    
    rolling_vol_std = rolling_vol_std.replace(0, 1e-9) # Block potential division by zero
    
    # Compute real-time statistical deviation (Z-Score)
    group['volatility_z_score'] = (group['daily_return'] - rolling_vol_mean) / rolling_vol_std
    
    # Isolate severe volume surges relative to standard behavior
    rolling_volm_mean = group['Volume'].rolling(window=50).mean()
    rolling_volm_std = group['Volume'].rolling(window=50).std().replace(0, 1e-9)
    group['volume_z_score'] = (group['Volume'] - rolling_volm_mean) / rolling_volm_std
    
    # Extract the absolute current active row matrix snapshot
    latest_state = group.iloc[-1]
    
    # An anomaly fires if market moves or volume pushes out past 3 standard deviations
    is_anomaly = (abs(latest_state['volatility_z_score']) > 3.0) or (latest_state['volume_z_score'] > 3.0)
    
    anomaly_records.append({
        'Symbol': symbol,
        'Current_Vol_ZScore': round(latest_state['volatility_z_score'], 2),
        'Current_Volm_ZScore': round(latest_state['volume_z_score'], 2),
        'Market_Anomaly_Active': 1 if is_anomaly else 0
    })

df_anomalies = pd.DataFrame(anomaly_records)
df_anomalies.to_csv(os.path.join(DATA_DIR, "market_anomalies.csv"), index=False)

print(f"SUCCESS: Anomaly tracking complete. Active market anomalies detected: {df_anomalies['Market_Anomaly_Active'].sum()}")

Deploying Statistical Anomaly Detection layers...
SUCCESS: Anomaly tracking complete. Active market anomalies detected: 1


In [23]:
print("Synthesizing unified advanced recommendations matrix...")

# Extract live end-of-timeline baseline records
latest_snapshots = df.sort_values('Date').groupby('Symbol').last().reset_index()

# Run concurrent dual-engine evaluations
X_inf = latest_snapshots[FEATURE_COLUMNS]
pred_cls = cls_model.predict(X_inf)
prob_cls = cls_model.predict_proba(X_inf)[:, 1]
pred_reg = reg_model.predict(X_inf)

latest_snapshots['ml_direction'] = pred_cls
latest_snapshots['ml_confidence'] = prob_cls
latest_snapshots['expected_5d_return'] = pred_reg

# Integrate risk parameters and anomaly vectors
df_risk = pd.read_csv(os.path.join(DATA_DIR, "risk_metrics.csv"))
master_intelligence_matrix = pd.merge(latest_snapshots, df_risk, on='Symbol')
master_intelligence_matrix = pd.merge(master_intelligence_matrix, df_anomalies, on='Symbol')

advanced_actions = []
detailed_justifications = []

# Generate explainable, conditional recommendations
for idx, row in master_intelligence_matrix.iterrows():
    symbol = row['Symbol']
    direction = row['ml_direction']
    conf = row['ml_confidence']
    exp_ret = row['expected_5d_return']
    risk_score = row['Risk_Score']
    anomaly = row['Market_Anomaly_Active']
    
    # Rule 1: Anomaly Interception Override
    if anomaly == 1:
        action = "HOLD"
        justification = (f"CRITICAL WARNING: Statistical market regime anomaly detected. Volatility Z-score sits at {row['Current_Vol_ZScore']:.1f}. "
                         f"Even with an expected return projection of {exp_ret*100:.1f}%, capital exposure is frozen for safety.")
    
    # Rule 2: Safe Buy Execution
    elif direction == 1 and risk_score < 60.0:
        action = "BUY"
        justification = (f"The asset shows solid upward momentum (Confidence: {conf*100:.1f}%) with a projected 5-day horizon return of +{exp_ret*100:.2f}%. "
                         f"Risk frameworks verify this security is within acceptable bounds (Risk Score: {risk_score:.1f}).")
    
    # Rule 3: High-Risk Exclusion Target
    elif direction == 1 and risk_score >= 60.0:
        action = "HOLD"
        justification = (f"ML model indicates an upward trend (+{exp_ret*100:.2f}% expected), but the asset belongs to a volatile "
                         f"historical risk category (Score: {risk_score:.1f}). Re-allocating exposure to a defensive posture.")
        
    # Rule 4: Sell Directive
    else:
        action = "SELL"
        justification = (f"Downward trend or flat performance expected over the next 5 sessions (Projected Change: {exp_ret*100:.2f}%). "
                         f"Pruning positions to avoid near-term capital drawdowns.")
        
    advanced_actions.append(action)
    detailed_justifications.append(justification)

master_intelligence_matrix['Action_Item'] = advanced_actions
master_intelligence_matrix['Quantitative_Justification'] = detailed_justifications

# Clean and filter output structure
final_fields = ['Symbol', 'Close', 'ml_confidence', 'expected_5d_return', 'Risk_Score', 'Risk_Category', 'Market_Anomaly_Active', 'Action_Item', 'Quantitative_Justification']
output_df = master_intelligence_matrix[final_fields].copy()

output_df.to_csv(os.path.join(DATA_DIR, "investment_recommendations.csv"), index=False)
print("\n" + "="*50)
print("SUCCESS: Phase 1 Advanced Analytics Core Finalized!")
print("Updated 'investment_recommendations.csv' written clean to workspace disk.")
print("="*50)

Synthesizing unified advanced recommendations matrix...

SUCCESS: Phase 1 Advanced Analytics Core Finalized!
Updated 'investment_recommendations.csv' written clean to workspace disk.
